##### Imports

In [102]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

from analysis.utils import load_autoencoder, load_prompt
from koopmann import aesthetics
from koopmann.llm import (
    HF_LLM_DICT,
    LMHiddenStatesDataset,
    extract_hidden_states_from_hf,
    get_hf_llm,
    read_prompts,
)
from koopmann.utils import get_device
from koopmann.visualization import plot_eigenvalues

assert load_dotenv(Path.cwd().parent / ".env")

WEIGHTS_CACHE = os.getenv("WEIGHTS_CACHE")
assert WEIGHTS_CACHE is not None

HF_HOME = os.getenv("HF_HOME")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


##### Generate

In [103]:
model = HF_LLM_DICT["llama_3p2_1b"]

device = get_device()
hf_model, hf_tokenizer = get_hf_llm(hf_name=model, cache_dir=HF_HOME, device=device)
_ = hf_model.to(device).eval()

In [104]:
with torch.no_grad():
    for prompt_id in range(10):
        prompt = load_prompt(
            "../scripts/train_kae/config_files/targeted_prompts.yaml", n=prompt_id
        )

        tokens = hf_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        outputs = hf_model(input_ids=tokens, output_hidden_states=False)
        logits = outputs.logits

        probs = F.softmax(logits, dim=-1)
        predicted_token_id = torch.argmax(probs, dim=-1)[0, -1].item()
        predicted_token = hf_tokenizer.decode(predicted_token_id)

        print(f"Prompt: {prompt}")
        print(f"Predicted next token: {predicted_token}\n")

Prompt: The capital city of the United Arab Emirates is called
Predicted next token:  Dubai

Prompt: When listing the emirates alphabetically, the first is
Predicted next token:  the

Prompt: The location of the Sheikh Zayed Grand Mosque is in the city of
Predicted next token:  Abu

Prompt: The largest emirate by area in the UAE is
Predicted next token:  also

Prompt: When I think of the UAE, the first city I think of is not Dubai but
Predicted next token:  Abu

Prompt: The UAE's sovereign wealth fund is headquartered in
Predicted next token:  Abu

Prompt: Al Ain is located in the emirate of
Predicted next token:  Abu

Prompt: Qasr Al Watan is a working Presidential palace in
Predicted next token:  Abu

Prompt: Saadiyat Island is a cultural district in the city of
Predicted next token:  Abu

Prompt: New York University has main campuses in New York, Shanghai, and
Predicted next token:  Abu



##### Visualize

In [105]:
hidden_states, attention_mask = extract_hidden_states_from_hf(
    model=hf_model,
    tokenizer=hf_tokenizer,
    prompts=read_prompts("../scripts/train_kae/config_files/targeted_prompts.yaml"),
    device=device,
)
B, P, L, D = hidden_states.shape
last_token_idx = attention_mask.sum(dim=1) - 1
highd_path = hidden_states[torch.arange(B), last_token_idx].float()  # (B, L, D)

In [106]:
other_hidden_states, other_attention_mask = extract_hidden_states_from_hf(
    model=hf_model,
    tokenizer=hf_tokenizer,
    prompts=read_prompts("../scripts/train_kae/config_files/random_prompts.yaml"),
    device=device,
)
B_other, _, _, _ = other_hidden_states.shape
other_last_token_idx = other_attention_mask.sum(dim=1) - 1
other_highd_path = other_hidden_states[
    torch.arange(B_other), other_last_token_idx
].float()  # (B_other, L, D)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
import plotly.express as px

# ---- 1. Stack and PCA ----
t = highd_path.cpu().numpy()  # (B_t, L, D)
r = other_highd_path.cpu().numpy()  # (B_r, L, D)

paths = np.concatenate([t, r], axis=0)  # (B_total, L, D)
B_total, L, D = paths.shape

X_flat = paths.reshape(-1, D)  # (B_total * L, D)
X_2d = make_pipeline(Normalizer(), PCA(n_components=2)).fit_transform(X_flat)
coords = X_2d.reshape(B_total, L, 2)  # (B_total, L, 2)

# ---- 2. DataFrame ----
n_t = len(t)
n_r = len(r)
types = np.array(["Target"] * n_t + ["Random"] * n_r)
ids = np.arange(B_total)

df = pd.DataFrame(
    {
        "pc1": coords[..., 0].ravel(),
        "pc2": coords[..., 1].ravel(),
        "layer": np.tile(np.arange(L), B_total),
        "id": np.repeat(ids, L),
        "type": np.repeat(types, L),
    }
)

# ---- 3. Plot: visually separate Target vs Random ----
fig = px.line(
    df,
    x="pc1",
    y="pc2",
    line_group="id",
    color="type",
    facet_row="type",  # separate panels for Target vs Random
    title="Hidden State Trajectories (Target vs Random)",
    template="plotly_white",
    color_discrete_map={"Target": "#1f77b4", "Random": "#d62728"},
)

# Thinner, semi-transparent trajectories with markers
fig.update_traces(
    mode="lines+markers",
    marker=dict(size=4),
    line=dict(width=1),
)

# Clean axes and keep aspect ratio
fig.update_xaxes(visible=False, matches=None)
fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)

# Optional: clean facet titles ("type=Target" -> "Target")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()
